In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import os

In [ ]:
INPUT_CSV = "/content/smart_meter_data (1).csv"
df = pd.read_csv(INPUT_CSV)

print("Dataset Loaded:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

Dataset Loaded: (5000, 7)
Columns: ['Timestamp', 'Electricity_Consumed', 'Temperature', 'Humidity', 'Wind_Speed', 'Avg_Past_Consumption', 'Anomaly_Label']
             Timestamp  Electricity_Consumed  Temperature  Humidity  \
0  2024-01-01 00:00:00              0.457786     0.469524  0.396368   
1  2024-01-01 00:30:00              0.351956     0.465545  0.451184   
2  2024-01-01 01:00:00              0.482948     0.285415  0.408289   
3  2024-01-01 01:30:00              0.628838     0.482095  0.512308   
4  2024-01-01 02:00:00              0.335974     0.624741  0.672021   

   Wind_Speed  Avg_Past_Consumption Anomaly_Label  
0    0.445441              0.692057        Normal  
1    0.458729              0.539874        Normal  
2    0.470360              0.614724        Normal  
3    0.576241              0.757044        Normal  
4    0.373004              0.673981        Normal  


In [ ]:
# Identify consumption column
# Try to guess column with "consumption" or "energy"
consumption_col = 'Avg_Past_Consumption'


values = df[consumption_col].dropna().astype(float).values

In [ ]:
# Benford’s Law

def benford_first_digit_stats(data):
    data = data[data > 0]
    first_digits = [int(str(int(x))[0]) for x in data if int(x) > 0]
    counts = Counter(first_digits)
    total = sum(counts.values())
    observed = np.array([counts[d]/total if d in counts else 0 for d in range(1,10)])
    expected = np.array([np.log10(1+1/d) for d in range(1,10)])
    mad = np.mean(np.abs(observed - expected))
    chi2_like = np.sum((observed-expected)**2 / expected)
    cdf_obs = np.cumsum(observed)
    cdf_exp = np.cumsum(expected)
    ks_stat = np.max(np.abs(cdf_obs - cdf_exp))
    return observed, expected, mad, chi2_like, ks_stat

observed, expected, mad, chi2_like, ks_stat = benford_first_digit_stats(values)
print("\nBenford’s Law Test:")
print("MAD:", mad, "Chi2-like:", chi2_like, "KS:", ks_stat)

plt.figure(figsize=(8,5))
plt.bar(range(1,10), observed, alpha=0.7, label="Observed")
plt.plot(range(1,10), expected, 'ro-', label="Expected")
plt.xlabel("First Digit")
plt.ylabel("Proportion")
plt.title("Benford’s Law Test")
plt.legend()
plt.savefig("benford_law.png")
plt.close()



Benford’s Law Test:
MAD: 0.1553266676302264 Chi2-like: 2.321928094887362 KS: 0.6989700043360187


In [ ]:
# Mantissa Arc Test

mantissas = np.mod(np.log10(values[values > 0]), 1)
angles = mantissas * 2 * np.pi
R = np.sqrt((np.sum(np.cos(angles))**2 + np.sum(np.sin(angles))**2)) / len(angles)
print("\nMantissa Arc Test: R =", R)


Mantissa Arc Test: R = 0.6591336612596721


In [ ]:
# Z-score and IQR Outliers

z_scores = (values - np.mean(values)) / np.std(values)
iqr = np.percentile(values,75) - np.percentile(values,25)
lower = np.percentile(values,25) - 1.5*iqr
upper = np.percentile(values,75) + 1.5*iqr
iqr_flags = ((values < lower) | (values > upper)).astype(int)

print("\nOutlier Counts:")
print("Z-score >3:", np.sum(np.abs(z_scores) > 3))
print("IQR flagged:", np.sum(iqr_flags))


Outlier Counts:
Z-score >3: 9
IQR flagged: 27


In [ ]:
# Relative Size Factor (RSF)

window = 24 if len(values) > 48 else max(3, len(values)//10)
rolling_mean = pd.Series(values).rolling(window, min_periods=1).mean().values
rsf = values / (rolling_mean + 1e-6)

In [ ]:
# Duplicates

dupes = df.duplicated().sum()
print("\nDuplicate Rows:", dupes)


Duplicate Rows: 0


In [ ]:
# Per-record forensic score

scores = []
for i,v in enumerate(values):
    score = 0
    if abs(z_scores[i]) > 3: score += 2
    if iqr_flags[i] == 1: score += 1
    if rsf[i] > 3: score += 2
    # weak benford flag
    if str(int(v))[0] in ['0','9']: score += 1
    scores.append(score)

df["forensic_score"] = scores
df["suspicious"] = (df["forensic_score"] >= 3).astype(int)

print("\nSuspicious Records Count:", df["suspicious"].sum())


Suspicious Records Count: 9


In [ ]:
# forensic_model.py
# Re-usable scoring function that mirrors the scoring used in the pipeline above.
# Import this in the dashboard or notebook UI.

import numpy as np
import pandas as pd

# Tunable parameters (keep same as pipeline)
ZSCORE_THRESH = 3.0
RSF_THRESH = 3.0
IQR_MULT = 1.5
WEIGHTS = {
    'z_flag': 2.0,
    'iqr_flag': 1.5,
    'rsf_flag': 2.0,
    'dup_flag': 2.0,
    'benford_weak': 0.5
}
MAX_POSSIBLE = sum(WEIGHTS.values())

def compute_population_stats(df, value_col='value', timestamp_col='timestamp', avg_past_col='avg_past'):
    """
    Compute population-level stats used for single-record scoring:
    returns dict with mean, std, rolling window size, quartiles etc.
    """
    stats = {}
    vals = pd.to_numeric(df[value_col], errors='coerce').dropna()
    stats['mean'] = float(vals.mean())
    stats['std'] = float(vals.std(ddof=0) if len(vals)>1 else 0.0)
    stats['q1'] = float(vals.quantile(0.25))
    stats['q3'] = float(vals.quantile(0.75))
    stats['iqr'] = stats['q3'] - stats['q1']
    # rolling window (use 24 or adapt)
    stats['rolling_window'] = 24
    # approximate rolling mean last value (for RSF) - we'll use population mean if rolling unavailable
    stats['pop_rolling_mean'] = float(vals.rolling(stats['rolling_window'], min_periods=1).mean().iloc[-1]) if len(vals)>0 else None
    return stats

def score_single_record(consumption,
                        avg_past=None,
                        population_stats=None,
                        is_duplicate=False):
    """
    Score a single input record and return (score_norm_0_10, raw_score, flags_dict)
    population_stats: dict as returned by compute_population_stats
    avg_past: historical average for that household if available (else None)
    is_duplicate: if you detected duplicate records externally (bool)
    """
    # defensive
    if population_stats is None:
        population_stats = {'mean':0.0, 'std':0.0, 'q1':0.0, 'q3':0.0, 'iqr':0.0, 'rolling_window':24, 'pop_rolling_mean':None}
    score = 0.0
    flags = {}
    # z-score flag
    stdev = population_stats.get('std', 0.0) or 0.0
    mean = population_stats.get('mean', 0.0)
    if stdev > 0:
        z = (consumption - mean) / stdev
    else:
        z = 0.0
    flags['zscore'] = z
    flags['z_flag'] = abs(z) > ZSCORE_THRESH
    if flags['z_flag']:
        score += WEIGHTS['z_flag']
    # iqr flag
    q1 = population_stats.get('q1',0.0)
    q3 = population_stats.get('q3',0.0)
    iqr = population_stats.get('iqr',0.0)
    lower = q1 - IQR_MULT*iqr
    upper = q3 + IQR_MULT*iqr
    flags['iqr_flag'] = (consumption < lower) or (consumption > upper)
    if flags['iqr_flag']:
        score += WEIGHTS['iqr_flag']
    # RSF flag (use avg_past if given else population rolling mean)
    base = None
    if avg_past is not None and avg_past > 0:
        base = avg_past
    else:
        base = population_stats.get('pop_rolling_mean', None)
    if base and base > 0:
        rsf_ratio = consumption / base
    else:
        rsf_ratio = 0.0
    flags['rsf_ratio'] = float(rsf_ratio)
    flags['rsf_flag'] = rsf_ratio > RSF_THRESH
    if flags['rsf_flag']:
        score += WEIGHTS['rsf_flag']
    # duplicate flag
    flags['dup_flag'] = bool(is_duplicate)
    if flags['dup_flag']:
        score += WEIGHTS['dup_flag']
    # weak benford: check first digit not 1 (weak signal)
    try:
        first_digit = int(str(int(abs(int(consumption))))[0])
    except Exception:
        first_digit = None
    flags['first_digit'] = first_digit
    flags['benford_weak'] = (first_digit != 1) if first_digit is not None else False
    if flags['benford_weak']:
        score += WEIGHTS['benford_weak']
    # normalized to 0-10
    score_norm = (score / MAX_POSSIBLE) * 10.0 if MAX_POSSIBLE>0 else 0.0
    flags['raw_score'] = float(score)
    flags['norm_score'] = float(score_norm)
    flags['suspicious'] = score_norm >= 4.0  # same threshold as pipeline
    return score_norm, score, flags


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
import pandas as pd
import numpy as np

# Input widgets
consumption_w = widgets.FloatSlider(min=0, max=1000, step=10, value=200, description="Consumption")
avg_past_w = widgets.FloatSlider(min=0, max=1000, step=10, value=150, description="Avg Past")
temp_w = widgets.FloatSlider(min=-10, max=50, step=1, value=25, description="Temp (°C)")
hum_w = widgets.FloatSlider(min=0, max=100, step=5, value=50, description="Humidity %")
wind_w = widgets.FloatSlider(min=0, max=50, step=1, value=5, description="Wind Speed")
dup_w = widgets.Checkbox(value=False, description="Duplicate Record?")
run_btn = widgets.Button(description="Run Forensic Check", button_style='info')

output = widgets.Output()
history = []  # store last 5 checks

def run_check(b):
    with output:
        clear_output(wait=True)
        # Run forensic scoring
        score_norm, raw_score, flags = score_single_record(
            consumption=consumption_w.value,
            avg_past=avg_past_w.value if avg_past_w.value>0 else None,
            population_stats=compute_population_stats(df, value_col="Avg_Past_Consumption"),
            is_duplicate=dup_w.value
        )

        # Adjust scoring with environment (bonus weights)
        if temp_w.value > 40 or temp_w.value < 5:  # extreme weather → suspicious
            score_norm += 1
            flags['env_flag'] = True
        else:
            flags['env_flag'] = False

        # Verdict
        verdict = "⚠️ SUSPICIOUS" if score_norm >= 4 else "✅ LEGAL"

        # Save to history
        history.append({
            "Consumption": consumption_w.value,
            "Avg_Past": avg_past_w.value,
            "Temp": temp_w.value,
            "Humidity": hum_w.value,
            "Wind": wind_w.value,
            "Verdict": verdict,
            "Score": round(score_norm,2)
        })
        if len(history) > 5:
            history.pop(0)

        # Dashboard display
        print("📌 Forensic Dashboard Result")
        print(f"Consumption: {consumption_w.value}, Avg Past: {avg_past_w.value}")
        print(f"Temp: {temp_w.value} °C, Humidity: {hum_w.value}%, Wind: {wind_w.value} m/s")
        print(f"\nRaw Score: {raw_score}/{MAX_POSSIBLE} | Normalized: {round(score_norm,2)}/10")
        print(f"\nVerdict: {verdict}")

        # Flags table
        display(pd.DataFrame([flags]))

        # Risk Gauge (0-10)
        plt.figure(figsize=(6,1.2))
        plt.barh([0], [score_norm], color="red" if score_norm>=4 else "green")
        plt.xlim(0,10)
        plt.yticks([])
        plt.title("Risk Score (0-10)")
        plt.show()

        # Comparison chart
        plt.figure(figsize=(5,3))
        plt.bar(["Consumption","Avg Past"], [consumption_w.value, avg_past_w.value])
        plt.title("Consumption vs Average Past")
        plt.ylabel("kWh")
        plt.show()

        # History Table
        print("\n📜 Last 5 Checks:")
        display(pd.DataFrame(history))

# Bind button
run_btn.on_click(run_check)

# Display full dashboard
display(widgets.VBox([consumption_w, avg_past_w, temp_w, hum_w, wind_w, dup_w, run_btn, output]))
